In [63]:
import os
from databricks.connect import DatabricksSession
from databricks.sdk.runtime import dbutils




In [64]:
project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.utilities import *

In [ ]:
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")
ENVIRONMENT = dbutils.widgets.get("environment")

ENV_CONFIG = {
    "dev": {
        "profile": "lab_4",
        "catalog": "dbr_dev",
        "storage_type": "managed",
    },
    "prod": {
        "profile": "lab_4_prod",
        "catalog": "dbr_dev",
        "storage_type": "external",
        "external_path": "abfss://yanquiel@dlspl21databricks.dfs.core.windows.net/landing",
    },
}
env_cfg = ENV_CONFIG[ENVIRONMENT]


if ENVIRONMENT not in ENV_CONFIG:
    raise ValueError(
        f"Environmet '{ENVIRONMENT}' not recognized. Valid options: {list(ENV_CONFIG.keys())}"
    )
env_cfg = ENV_CONFIG[ENVIRONMENT]

IS_LOCAL = os.environ.get("DATABRICKS_RUNTIME_VERSION") is None

dbutils.widgets.text("cluster_id", "", "Cluster ID (vacío = serverless)")
CLUSTER_ID = dbutils.widgets.get("cluster_id")

if IS_LOCAL:
    builder = DatabricksSession.builder.profile(env_cfg["profile"])
    spark = (builder.clusterId(CLUSTER_ID) if CLUSTER_ID else builder.serverless()).getOrCreate()


Box(children=(Label(value='Environment'), Dropdown(options=('dev', 'prod'), value='dev')))

Box(children=(Label(value='Cluster ID (vacío = serverless)'), Text(value='')))

In [58]:
CATALOG = env_cfg["catalog"]

In [59]:
VOLUME_BASE = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/landing"

LANDING_MENU_PATH = f"{VOLUME_BASE}/menu"
LANDING_ORDERS_PATH = f"{VOLUME_BASE}/orders"  

ORDERS_SCHEMA_LOCATION = f"{VOLUME_BASE}/_checkpoints/orders/schema"
ORDERS_CHECKPOINT_LOCATION = f"{VOLUME_BASE}/_checkpoints/orders/checkpoint"

BRONZE_MENU_FULL_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_MENU_TABLE}"
BRONZE_ORDER_FULL_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_ORDER_TABLE}"

In [60]:
BOOTSTRAP_SERVERS = "pkc-56d1g.eastus.azure.confluent.cloud:9092"
TOPIC_NAME = "orders_event"